# Загрузка и объединение данных

Ноутбук выполняет загрузку данных из всех источников и их объединение:

1. **Ручная разметка** — `dataset.json` + экспорты чатов (ham)
2. **Discord** — `dataset_discord.json` (spam)
3. **tg-spam (umputun)** — `spam_samples_umputun.txt` (spam)
4. **lols.bot** — `parsed_lols_2024.txt` (spam)
5. **Незарегистрированный спам** — из SQLite базы бота

После объединения проверяются конфликты меток и результат сохраняется в `data/interim/raw_combined.csv`.

## Импорт необходимых библиотек

Используются `pandas`, `numpy` и модули проекта:
- `src.data.loaders` — загрузка JSON и TXT файлов
- `src.data.preprocessing` — предобработка текста
- `src.features.extractors` — извлечение числовых признаков
- `src.config` — пути к директориям данных

In [1]:
import json
import os
import re
import sqlite3
import sys
import numpy as np
import pandas as pd

try:
    _project_root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    _cwd = os.getcwd()
    _project_root = _cwd
    while _project_root != '/' and not os.path.isdir(os.path.join(_project_root, 'src')):
        _project_root = os.path.dirname(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.data.loaders import load_json, load_txt_lines
from src.data.preprocessing import preprocess_text, normalize_unicode, strip_html
from src.features.extractors import (
    count_emojis,
    count_newlines,
    count_whitespaces,
    count_links,
    count_tags,
    capital_ratio,
    count_punctuation,
    count_digits,
    avg_word_length,
    word_count,
    unique_word_ratio,
    repeat_char_ratio,
    count_phone_numbers,
    has_crypto_mention,
    count_exclamation,
    url_ratio,
    count_html_tags,
    has_markdown_formatting,
    emoji_diversity,
)
from src.config import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR, PROJECT_ROOT, CHAT_EXPORTS_DIR

from copy import deepcopy

## Первичная подготовка данных

Данные собираются из четырёх источников и объединяются в один датафрейм.

### Чтение файлов подготовленных вручную

Источники с ручной разметкой: `dataset.json` (чат "ЧАТ_СТАНКИН Абитуриент")

`data/dataset.json` – `.json` файл структуры `[{"text": "blablabla", "label": 0}]`, где ключ `text` представляет текстовое значение сообщения, а ключ `label` – класс сообщения (0: безопасное, 1: опасное).
Сообщения были собраны из самого чата "ЧАТ_СТАНКИН Абитуриент".

In [2]:
_dataset_path = RAW_DIR / 'dataset.json'
df_manual = pd.read_json(_dataset_path)
new_records = []
for _subdir in sorted(os.listdir(CHAT_EXPORTS_DIR)):
    _result_path = os.path.join(CHAT_EXPORTS_DIR, _subdir, 'result.json')
    if not os.path.exists(_result_path):
        continue
    with open(_result_path, 'r', encoding='utf-8') as _f:
        _data = json.load(_f)
    for _msg in _data.get('messages', []):
        if _msg.get('type') == 'service':
            continue
        _text_entities = _msg.get('text_entities')
        if _text_entities:
            _msg_text = ''.join((e.get('text', '') for e in _text_entities))
        elif isinstance(_msg.get('text'), str):
            _msg_text = _msg['text']
        else:
            continue
        _msg_text = _msg_text.strip()
        if _msg_text:
            new_records.append({'text': _msg_text, 'label': 0})
if new_records:
    df_new = pd.DataFrame(new_records)
    df_manual = pd.concat([df_manual, df_new], ignore_index=True)
    print(f'Добавлено {len(new_records)} сообщений из экспортов чатов')
    print(f'Всего в dataset.json (в памяти): {len(df_manual)} строк')
    print('Запись на диск — после проверки конфликтов меток')
else:
    print('Нет новых сообщений из экспортов чатов')

Добавлено 97928 сообщений из экспортов чатов
Всего в dataset.json (в памяти): 531804 строк
Запись на диск — после проверки конфликтов меток


In [3]:
df_manual.head()

,text,label
0,Добрый день! Отличается ли перечень необходимы...,0
1,Узбекистан. Рассматриваются обе формы,0
2,"Здравствуйте, а как проходит поступление после...",0
3,Спасибо большое за ответ!,0
4,"Здравствуйте, а когда будет день открытых двер...",0


`data/dataset_discord.json` – `.json` файл структуры `[{"text": "blablabla", "label": 0}]`, где ключ `text` представляет текстовое значение сообщения, а ключ `label` – класс сообщения (~~0: безопасное~~, 1: опасное).
Сообщения были собраны из чатов стороннего Discord* сервера.

> _Discord* – запрещенная социальная сеть в Российской Федерации._

In [4]:
df_discord = pd.read_json(RAW_DIR / 'dataset_discord.json')

In [5]:
df_discord.head()

,text,label
0,# _**Лучший спам-бот!!! **_\n**залетай к нам н...,1
1,Лучший кряк нурсултана! https://wdfiles.ru/1tf...,1
2,https://oxy.name/d/pddi я заболел раокм гамна ...,1
3,Забирайте S.T.A.L.K.E.R 2 предзаказ на свой St...,1
4,https://only-fans.uk/morphe_ya\nOnlyFans UK - ...,1


### Чтение файлов, предоставленных сторонними ресурсами

Внешние источники спам-сообщений: tg-spam (umputun) и lols.bot.

#### Umputun's tg-spam Dataset

`data/spam_samples_umputun.txt` – `.txt` файл с набором спам-сообщений.
Файл был получен с [umputun/tg-spam GitHub](https://github.com/umputun/tg-spam/tree/master/data)

In [6]:
spam_tg_umputun = load_txt_lines(EXTERNAL_DIR / 'spam_samples_umputun.txt')

In [7]:
for _i in range(len(spam_tg_umputun)):
    spam_tg_umputun[_i] = spam_tg_umputun[_i].strip().rstrip()

In [8]:
spam_tg_umputun

["Use this link and register your account now start earning in cryptocurrency and testify to others Https://accetgrowth.ltd Use this link and register your account now start earning in cryptocurrency and testify to others https://tinyurl.com/yrd86r5b Your investment is safe and secure with this transparent and legit platform https://tinyurl.com/yrd86r5b Don't miss this opportunity to earn with this platform and be financially stable click on the link to join https://urlz.fr/orYP Trusted and legit investment platform no hidden charges https://tinyurl.com/yrd86r5b",
 'Globalstockoptions is one of the best trade broker trade platform I have been trading with them more than 9 months and I have got withdrawal ib commission more than $350000, I have more than 250 plus clients today those trading with Globalstockoptions and best part no one is calling me for any issues or not even for any update. I am so grateful and thankful to Globalstockoption, finally we have broker who do not have issues

Приведение таблицы (датафрейма) к виду `['text', 'label']`

In [9]:
df_tg_umputun = pd.DataFrame()
df_tg_umputun['text'] = spam_tg_umputun
df_tg_umputun['label'] = [1] * len(spam_tg_umputun)

In [10]:
df_tg_umputun.head()

,text,label
0,Use this link and register your account now st...,1
1,Globalstockoptions is one of the best trade br...,1
2,Now I believe in what people say about you is ...,1
3,https://t.me/+jS6cJeri0ug5MDkx This is very ...,1
4,Investing in a genuine company with great inve...,1


#### LOLS Bot Data 2024

`data/parsed_lols_2024.txt` – `.txt` файл с набором спам-сообщений.
Файл был получен с [lols.bot](https://lols.bot/2024/)

In [11]:
parsed_lols_2024 = load_txt_lines(EXTERNAL_DIR / 'parsed_lols_2024.txt')

Базовая очистка: первый <br/> убирается, остальные заменяются на \n

In [12]:
for _i in range(len(parsed_lols_2024)):
    parsed_lols_2024[_i] = parsed_lols_2024[_i].replace('<br/>', '', 1).strip().replace('<br/>', '\n')

In [13]:
len(parsed_lols_2024)

34381

In [14]:
df_lols_2024 = pd.DataFrame()
df_lols_2024['text'] = parsed_lols_2024
df_lols_2024['label'] = [1] * len(parsed_lols_2024)

In [15]:
df_lols_2024.head(3)

,text,label
0,Нужны срочно ребятa нa офис 8000 4-6чaсов сроч...,1
1,Нужeн пoмoщник нa cклaд зп в день от 3000pyб п...,1
2,🔤🔤🔤🔤🔤🔤<br><br> НЕТ 18 ЛЕТ — НЕТ ДЕНЕГ <br...,1


### Извлечение незарегистрированного спама

Выявляются спам-сообщения, которые были удалены из чата вручную
модераторами, но не были пойманы ботом автоматически.

Алгоритм:
1. Берутся все сообщения, которые бот видел (таблица `collected_message`).
2. Из них вычитаются сообщения из всех экспортов чатов (остались в чате = безопасные).
3. Вычитаются сообщения из таблицы `spam_message` (уже пойманный спам).
4. Остаток — сообщения, удалённые вручную, с высокой вероятностью являющиеся спамом.

Если файл `data/interim/unregistered_spam.json` уже существует,
он загружается без перезаписи (для ручной проверки и редактирования).
Иначе файл создаётся автоматически.

In [16]:
DB_PATH = os.path.join(PROJECT_ROOT, 'instance', 'db.sqlite')
UNREGISTERED_OUTPUT = EXTERNAL_DIR / 'unregistered_spam.json'

In [17]:
df_unregistered = pd.DataFrame(columns=['text', 'label'])

In [18]:
if os.path.exists(UNREGISTERED_OUTPUT):
    df_unregistered_1 = pd.read_json(UNREGISTERED_OUTPUT)
    if 'label' not in df_unregistered_1.columns:
        df_unregistered_1['label'] = 1
    print(f'Загружено из существующего файла: {len(df_unregistered_1)} строк')
elif os.path.exists(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    df_collected = pd.read_sql_query('SELECT DISTINCT message_text FROM collected_message', conn)
    df_spam = pd.read_sql_query('SELECT DISTINCT message_text FROM spam_message', conn)
    conn.close()
    chat_texts = set()
    for _subdir in sorted(os.listdir(CHAT_EXPORTS_DIR)):
        _result_path = os.path.join(CHAT_EXPORTS_DIR, _subdir, 'result.json')
        if not os.path.exists(_result_path):
            continue
        with open(_result_path, 'r', encoding='utf-8') as _f:
            _data = json.load(_f)
        for _msg in _data.get('messages', []):
            if _msg.get('type') == 'service':
                continue
            _text_entities = _msg.get('text_entities')
            if _text_entities:
                _msg_text = ''.join((e.get('text', '') for e in _text_entities))
            elif isinstance(_msg.get('text'), str):
                _msg_text = _msg['text']
            else:
                continue
            _msg_text = _msg_text.strip()
            if _msg_text:
                chat_texts.add(_msg_text)
    collected_texts = set(df_collected['message_text'].tolist())
    spam_texts = set(df_spam['message_text'].tolist())
    unregistered = collected_texts - chat_texts - spam_texts
    print(f'Собрано ботом:        {len(collected_texts)}')
    print(f'Экспорт чатов:        {len(chat_texts)}')
    print(f'Спам (поймано):       {len(spam_texts)}')
    print(f'Незарегистрировано:   {len(unregistered)}')
    if unregistered:
        df_unregistered_1 = pd.DataFrame([{'text': t, 'label': 1} for t in unregistered if t.strip()])
        with open(UNREGISTERED_OUTPUT, 'w', encoding='utf-8') as _f:
            json.dump(df_unregistered_1.to_dict('records'), _f, ensure_ascii=False, indent=2)
        print(f'Сохранено в {UNREGISTERED_OUTPUT}')
else:
    print('БД и файл не найдены — пропускаем')
print(f'\nИтого незарегистрированного спама: {len(df_unregistered_1)} строк')

Загружено из существующего файла: 36 строк

Итого незарегистрированного спама: 36 строк


### Объединение подходящих файлов

Все источники объединяются в один датафрейм `df`.
Перед объединением проверяются конфликты меток: если один и тот же
текст встречается с разными `label` в разных источниках — ячейка
выводит конфликты с указанием файла-источника и прерывает выполнение.

Также только после успешной проверки `dataset.json` записывается на диск
(с добавленными текстами из экспортов чатов).

In [19]:
print(f'До объединения: {len(df_manual)} строк')

До объединения: 531804 строк


In [20]:
df_manual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 531804 entries, 0 to 531803
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    531804 non-null  object
 1   label   531804 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 8.1+ MB


In [21]:
df_manual['_source'] = 'dataset.json'
df_tg_umputun['_source'] = 'spam_samples_umputun.txt'
df_discord['_source'] = 'dataset_discord.json'
df_lols_2024['_source'] = 'parsed_lols_2024.txt'
df_unregistered_1['_source'] = 'unregistered_spam.json'

Временное объединение для проверки конфликтов до создания df

In [22]:
_tmp = pd.concat([df_manual, df_tg_umputun, df_discord, df_lols_2024, df_unregistered_1], ignore_index=True)
_tmp = _tmp[_tmp['text'].notna()]
_tmp = _tmp[_tmp['text'].str.strip() != '']
_tmp = _tmp[_tmp['text'].str.len() >= 3]

In [23]:
label_groups = _tmp.groupby('text')['label'].nunique()
conflicts = label_groups[label_groups > 1]

In [24]:
if len(conflicts) > 0:
    print(f'ОШИБКА: найдено {len(conflicts)} текстов с конфликтными метками!')
    print('Объединение отменено. Исправьте метки в исходных файлах и перезапустите ячейки до этой включительно.\n')
    for txt in conflicts.index:
        rows = _tmp[_tmp['text'] == txt]
        labels = rows['label'].tolist()
        sources = rows['_source'].tolist()
        preview = txt[:80].replace('\n', ' ')
        print(f'  "{preview}"')
        for lbl, src in zip(labels, sources):
            print(f'    label={lbl}  источник={src}')
        print()
    raise ValueError(f'Найдено {len(conflicts)} конфликтных меток. Исправьте исходные данные.')

In [25]:
# Запись dataset.json на диск — только после успешной проверки конфликтов
_dataset_path = RAW_DIR / 'dataset.json'
_df_save = df_manual.drop(columns=['_source'], errors='ignore')
_df_save.to_json(_dataset_path, orient='records', force_ascii=False, indent=2)
print(f'dataset.json записан: {len(_df_save)} строк')

dataset.json записан: 531804 строк


In [26]:
df = pd.concat([df_manual, df_tg_umputun, df_discord, df_lols_2024, df_unregistered_1], ignore_index=True)
print(f'Всего строк: {len(df)} (из них незарегистрированного спама: {len(df_unregistered_1)})')

Всего строк: 566420 (из них незарегистрированного спама: 36)


## Сохранение объединённого датасета

Результат сохраняется в `data/interim/raw_combined.csv` для использования в следующем ноутбуке.

In [27]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_DIR / 'raw_combined.csv', index=False)
print(f'Сохранено: {len(df)} строк в {INTERIM_DIR / "raw_combined.csv"}')

Сохранено: 566420 строк в /home/sophrosyne/STANKIN_AntiSpam_Bot/data/interim/raw_combined.csv
